### Load PDF file

In [3]:
import os
if os.path.exists("../LLM/NovaS.pdf"):
    print("File exists")
else:
    print("File does not exist")

File exists


### Load Libraries

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

C:\Users\v-kpurwar\AppData\Local\Temp\ipykernel_40924\1491740739.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


True

#### Step 0: Convert PDF into text

In [8]:
docs = PyPDFLoader("../LLM/NovaS.pdf").load()

full_text = "\n".join([doc.page_content for doc in docs])
full_text 
## convert pdf into text

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 

### Split into Semantic chunks

In [18]:
chunker = SemanticChunker(
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2"),
    breakpoint_threshold_type = "percentile",
    breakpoint_threshold_amount = 60,
)

semantic_chunks = chunker.create_documents([full_text])
# When we pass full_text as a string, Python iterates it character-by-character,
# so each character becomes its own "document" → hence single-character chunks.
semantic_chunks

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting 

In [19]:
len(semantic_chunks)  # Check the number of chunks created

28

### Create Embedding and store them into VectorDB

In [20]:
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [21]:
chroma_db = Chroma.from_documents(semantic_chunks, embed_model, persist_directory="./chroma_db_semantic")
# created the database in the local directory. The database is created in the form of a folder called chroma_db_semantic.
# This folder contains the embeddings of the chunks. We can use this database to retrieve the chunks based on the query.